In [1]:
# Load packages
import os.path as op
import glob
import shutil
import mne
import numpy as np
import pandas as pd
from mne_bids import (
    BIDSPath,
    make_dataset_description,
    print_dir_tree,
    read_raw_bids,
    write_raw_bids,
    get_anonymization_daysback,
)

In [2]:
# Make sub id list
subject_ids = np.arange(1,36)
# Define data paths
root_dir = 'C:/Users/mvmigem/Documents/data/project_2/'
raw_data_dir = op.join(root_dir,"raw")
bids_root = op.join(root_dir,"bids")
# get individual paths
sub_paths = glob.glob(raw_data_dir + "/sub*")

In [3]:
# Empty if not
if op.exists(bids_root):
    shutil.rmtree(bids_root)

In [4]:
# Event id's
event_id = {
    'start_trial':99,  
    'pos1/seq1':11, 'pos1/seq2':12, 
    'pos1/seq3':13, 'pos1/seq4':14, 
    'pos2/seq1':21, 'pos2/seq2':22,
    'pos2/seq3':23, 'pos2/seq4':24,
    'pos3/seq1':31, 'pos3/seq2':32,
    'pos3/seq3':33, 'pos3/seq4':34,
    'pos4/seq1':41, 'pos4/seq2':42,
    'pos4/seq3':43, 'pos4/seq4':44,
    }
event_metadata_id = { 'lab': 'system used', 'mode':'pilor or default', 
                      'eye_tracking':'indiacte if eye-tracking was eneabled', 'participant': 'subject identification', 'gender':'gender',
                      'age':'in years', 'handed':'dominant hand', 'intruct_lang':'instruction language', 
                      'located_quad':'quadrant signalled indentified as having the smallest C1 by the localiser see position for code', 
                      'trial':'trial number', 'block': 'block number', 'start_position':'position of first stimulus in trial, see position for code', 
                      'rotation_odd':'the direction that was less likely (odd) for this subject', 
                      'clockwise_key':'key to indicate clockwise for this subject', 'anticlockwise_key':'key to indicate anticlockwise for this subject', 
                      'angle_odd':'the line orientation that was less likely (odd) for this subject ', 
                      'angle_horz_key':'key to indicate horizontal for this subject', 'angle_vert_key':'key to indicate vertical for this subject', 
                      'trial_direction': 'rotation direction for this trial, 0: clockwise , 1:anticlockwise', 
                      'angle': 'line orientation for this trial, 0 horizontal, 1 vertical', 'target_trials': 'whether trial was taget, 0: no, 1: yes', 
                      't_trial':'time of start of trial relative to experiment start (s)','experiment_time_s': 'time relative to experiment start (s)',
                      't_stim':'time of stimulus relative to trial start (s)', 'position':'position of stimulus on screen, topleft:1, top right:2, bottom right:3,bottom left: 4 ', 
                      'sequence':'order of stimulus', 'eeg_trigger':'trigger should match value', 'key_data':'keypress metadata', 
                      'rotation_prediction':'whether rotation is regular or odd', 'angle_prediction':'whether orientation is regular or odd', 
                      'task':'which feature particpants need to attend', 'accuracy':' accuracy of trial', 'rt':'reaction time' }

values_to_keep = [99,11,12,13,14,21,22,23,24,31,32,33,34,41,42,43,44]
montage = mne.channels.make_standard_montage('biosemi64')

In [10]:
raw_list = list()
events_list = list()
bids_list = list()
# Select montage

# iterate over subjects
for i,sub in enumerate(subject_ids):
    # EEG part
    raw_path = op.join(sub_paths[i],'eeg','*.bdf')
    raw_fname = glob.glob(raw_path) [0]
    raw = mne.io.read_raw_bdf(raw_fname, preload = False)
    raw_list.append(raw)
    # specify power line frequency, and ref as required by BIDS
    raw.info['line_freq'] = 50

    fix_chans = {'EXG1':'eye_above','EXG2':'eye_below',
             'EXG3':'eye_left','EXG4':'eye_right',
             'EXG5':'M1','EXG6':'M2'}
    raw.rename_channels(fix_chans)
    raw.drop_channels(['EXG7', 'EXG8'])
    raw.set_channel_types({'M1':'misc', 'M2':'misc',
                       'eye_above':'eog', 'eye_below':'eog',
                       'eye_left':'eog', 'eye_right': 'eog'})
    # This dict is to rename the channel names to fit the montage
    mon_chnames = montage.ch_names
    raw_chnames = raw.info['ch_names']
    rename_channels = dict(zip(raw_chnames[:64],mon_chnames))
    raw.rename_channels(rename_channels)
    ## Get events and combine with behavior data
    filtered_event = mne.find_events(raw, shortest_event=1)

    events = filtered_event[np.isin(filtered_event[:, -1], values_to_keep)]
    # Behaviour part
    behav_path = op.join(root_dir,'aligned_behav',f'aligned-behaviour-sub-{sub:02}.csv')
    behav_fname = glob.glob(behav_path) [0]
    behav = pd.read_csv(behav_fname)
    # Clean a little
    behav = behav.loc[:, ~behav.columns.str.contains('^Unnamed')]
    behav = behav.loc[:, ~behav.columns.str.contains('this')]
    behav = behav.loc[:, ~behav.columns.str.contains('LocalTime')]
    behav = behav.loc[:, ~behav.columns.str.contains('notes')]
    # events dataframe
    # events_df = pd.DataFrame({
    # 'onset': events[:, 0] / raw.info['sfreq'],  # samples → seconds
    # 'duration': 0,                                # or real durations if you have them
    # 'trial_type': events[:, 2]                    # event codes
    # })
    # events_df = pd.concat([events_df.reset_index(drop=True), 
    #                     behav.reset_index(drop=True)], axis=1)
    # --- Write BIDS ---
    bids_path = BIDSPath(
        subject= f"{sub:02}",
        task = "lineMotionDiscrimination",
        root = bids_root
    )
    bids_path_out = write_raw_bids(
        raw,
        bids_path=bids_path,
        events=events,
        event_id=event_id,
        event_metadata= behav,
        extra_columns_descriptions= event_metadata_id,
        montage= montage,
        anonymize = anonymize,
        overwrite=True
    )
    


Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-01\eeg\main_01.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Finding events on: Status


C:\Users\mvmigem\AppData\Local\Temp\ipykernel_11360\2264715107.py:21: RuntimeWarning: The unit for channel(s) M1, M2 has changed from V to NA.
  raw.set_channel_types({'M1':'misc', 'M2':'misc',


Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
4530 events found on stim channel Status
Event IDs: [   11    13    22    24    31    33    42    44    99 65536 65635 65789]
Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-01\eeg\main_01.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Not fully anonymizing info - keeping his_id, sex, and hand info
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\README'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\participants.tsv'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\participants.json'...
Used Annotations descriptions: [np.str_('pos1/seq1'), np.str_('pos1/seq3'), np.str_('pos2/seq2'), np.str_('pos2/seq4'), np.str_('pos3/seq1'), np.str_('pos3/seq3'), np.str_('pos4/seq2'), np.str_('pos4/seq4'), np.str_('start_trial')]


C:\Users\mvmigem\AppData\Local\Temp\ipykernel_11360\2264715107.py:56: RuntimeWarning: Coordinate frame could not be inferred from the raw object and the BIDSPath.space was none, skipping the writing of channel positions
  bids_path_out = write_raw_bids(


Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\sub-01\eeg\sub-01_task-lineMotionDiscrimination_events.tsv'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\sub-01\eeg\sub-01_task-lineMotionDiscrimination_events.json'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\dataset_description.json'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\sub-01\eeg\sub-01_task-lineMotionDiscrimination_eeg.json'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\sub-01\eeg\sub-01_task-lineMotionDiscrimination_channels.tsv'...
Copying data files to sub-01_task-lineMotionDiscrimination_eeg.bdf


...

Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
4530 events found on stim channel Status
Event IDs: [   11    13    22    24    31    33    42    44    99 65536 65635 65789]
Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-35\eeg\main_35.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Not fully anonymizing info - keeping his_id, sex, and hand info
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\participants.tsv'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\participants.json'...
Used Annotations descriptions: [np.str_('pos1/seq1'), np.str_('pos1/seq3'), np.str_('pos2/seq2'), np.str_('pos2/seq4'), np.str_('pos3/seq1'), np.str_('pos3/seq3'), np.str_('pos4/seq2'), np.str_('pos4/seq4'), np.str_('start_trial')]


C:\Users\mvmigem\AppData\Local\Temp\ipykernel_11360\2264715107.py:56: RuntimeWarning: Coordinate frame could not be inferred from the raw object and the BIDSPath.space was none, skipping the writing of channel positions
  bids_path_out = write_raw_bids(


Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\sub-35\eeg\sub-35_task-lineMotionDiscrimination_events.tsv'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\sub-35\eeg\sub-35_task-lineMotionDiscrimination_events.json'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\dataset_description.json'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\sub-35\eeg\sub-35_task-lineMotionDiscrimination_eeg.json'...
Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\sub-35\eeg\sub-35_task-lineMotionDiscrimination_channels.tsv'...
Copying data files to sub-35_task-lineMotionDiscrimination_eeg.bdf
Not fully anonymizing info - keeping his_id, sex, and hand info


C:\Users\mvmigem\AppData\Local\Temp\ipykernel_11360\2264715107.py:56: RuntimeWarning: EDF/EDF+/BDF files contain two fields for recording dates.Due to file format limitations, one of these fields only supports 2-digit years. The date for that field will be set to 85 (i.e., 1985), the earliest possible date. The true anonymized date is stored in the scans.tsv file.
  bids_path_out = write_raw_bids(


Writing 'C:\Users\mvmigem\Documents\data\project_2\bids\sub-35\sub-35_scans.tsv'...
Wrote C:\Users\mvmigem\Documents\data\project_2\bids\sub-35\sub-35_scans.tsv entry with eeg\sub-35_task-lineMotionDiscrimination_eeg.bdf.


In [6]:
raw_list = list()
events_list = list()
bids_list = list()
# Select montage

# iterate over subjects
for i,sub in enumerate(subject_ids):
    # EEG part
    raw_path = op.join(sub_paths[i],'eeg','*.bdf')
    raw_fname = glob.glob(raw_path) [0]
    raw = mne.io.read_raw_bdf(raw_fname, preload = False)
    raw_list.append(raw)

Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-01\eeg\main_01.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-02\eeg\main_02.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-03\eeg\main_03.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-04\eeg\main_04.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-05\eeg\main_05.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project

In [8]:
daysback_min, daysback_max = get_anonymization_daysback(raw_list)

In [ ]:
anonymize=dict(daysback=daysback_min + xxxx,keep_his=True)

In [21]:
# Fix participants.tsv manually

# I have a demogaphics file
demos = pd.read_csv(root_dir + 'demographics.csv',index_col=0)

bids_participants = pd.read_csv(root_dir + '/bids/participants.tsv', sep= '\t')
bids_participants['age'] = demos['age']
bids_participants['sex'] = demos['gender']
bids_participants['hand'] = demos['handed']

bids_participants.to_csv(op.join(root_dir, 'bids', 'participants.tsv'), sep='\t',index = False)

In [19]:
print(demos.index.tolist())
print(bids_participants.index.tolist())


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-20', 'sub-21', 'sub-22', 'sub-23', 'sub-24', 'sub-25', 'sub-26', 'sub-27', 'sub-28', 'sub-29', 'sub-30', 'sub-31', 'sub-32', 'sub-33', 'sub-34', 'sub-35']


In [ ]:
import osfclient

